# Missing Values Imputation

This notebook implements a comprehensive missing value imputation strategy for the student dropout dataset.

## Notebook Overview

1. **Setup & Data Loading**: Import libraries and load preprocessed data
2. **Exploratory Analysis**: Analyze missing value patterns and cohort distributions
3. **Imputation Strategy**: Apply structured imputation based on missingness type
4. **Validation**: Verify imputation results
5. **Export**: Save the processed dataset

## Imputation Strategy Summary

- **Structural missingness**: Preserved (e.g., no orientation activity → NaT/0)
- **Academic performance**: Sat_Exam flags + median imputation for students who sat exams
- **Distances**: Median imputation with missing flags
- **Demographics**: Median/mode imputation with flags
- **Dates**: Cohort-aware median imputation for exam dates

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
import warnings

# Configure display and warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None


## 1. Setup & Data Loading

In [ ]:
# Load preprocessed data
current_user = os.getlogin()
DATA_DIR = Path(f"/home/{current_user}/local-share")
OUT_DIR = DATA_DIR / "processed"

data = pd.read_csv(OUT_DIR / "handled_low_frequency_categories.csv", 
                   sep=None, engine="python", encoding="utf-8-sig")

print(f"Dataset shape: {data.shape}")
data.head()

In [ ]:
# Quick data overview
print(f"Columns: {len(data.columns)}")
data.info()

## 2. Exploratory Analysis

### 2.1 Cohort Distribution

Before performing cohort-aware imputation, we verify that each cohort has sufficient observations for reliable statistics.

In [ ]:
# Analyze cohort distribution
year_col = "sdo5_degree_COLLEGEJAAR"
data[year_col] = pd.to_numeric(data[year_col], errors="coerce")

# Display counts and percentages
cohort_counts = data[year_col].value_counts(dropna=False).sort_index()
cohort_pct = (cohort_counts / len(data) * 100).round(2)

print("Cohort Distribution:")
print(pd.DataFrame({'Count': cohort_counts, 'Percentage': cohort_pct}))

# Visualize distribution
plt.figure(figsize=(10, 5))
cohort_counts.plot(kind="bar", color='steelblue')
plt.title("Distribution of Student Start Years (Cohorts)", fontsize=14, fontweight='bold')
plt.xlabel("Cohort Year")
plt.ylabel("Number of Students")
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Result**: Each cohort has sufficient observations for cohort-wise imputation.

### 2.2 Missing Values Analysis

In [ ]:
# Analyze missing values
nan_summary = pd.DataFrame({
    'NaN Count': data.isna().sum(),
    'NaN %': (data.isna().sum() / len(data) * 100).round(2)
}).sort_values('NaN Count', ascending=False)

# Display all columns with missing values
print("All Columns with Missing Values:")
print(nan_summary[nan_summary['NaN Count'] > 0])

## 3. Imputation Strategy

### Missing Value Categories

1. **Orientation participation** (~64% missing, structural)
   - Numeric counts → fill with 0
   - Dates → keep as NaT (students never attended)

2. **Academic performance** (9-24% missing)
   - Average grades (A/B): Create `Sat_Exam` flags + median imputation
   - Percentages/Potentials: Fill with 0 (no exam activity)

3. **Distances** (3-9% missing)
   - Median imputation + missing flags

4. **Dates with partial missingness**
   - Previous exam date → **cohort-wise median** (by COLLEGEJAAR)
   - SKC date → keep NaT (structural)

5. **SDO78 (100 Dagen Monitor) questionnaire** (structural for non-responders, partial for others)
   - `sdo78_response_type` already encodes responder status (Complete-responder, Partial-responder, Non-responder) — this column acts as the responder indicator and is **not imputed**. We do however regard all students who have to record for sdo78 also Non-responders.
   - Numeric sdo78 question columns (HU*, L* prefixed) and aggregated scales (Support_Avg, Belonging_Avg): cohort-wise **median imputation** by COLLEGEJAAR for all students (including non-responders), to allow model training without excluding non-responders
   - Rationale: non-responders show ~36% dropout vs ~19% for responders; imputing with cohort median instead of excluding them retains this signal while avoiding data leakage from the response status into question-level features
   - The `sdo78_*_missing_flag` columns are dropped downstream in notebook 10 because `sdo78_response_type` already captures the missingness structure comprehensively

In [ ]:
def impute_all(data: pd.DataFrame) -> pd.DataFrame:
    """
    Apply comprehensive missing value imputation strategy.
    
    Returns:
        DataFrame with imputed values and missing indicator flags.
    """
    df = data.copy()

    # Define column groups
    orientation_count_cols = [
        "sdo2_orientation_Number_of_Event_Types",
        "sdo2_orientation_Number_of_Events_Attended",
    ]
    orientation_date_cols = [
        "sdo2_orientation_First_Event_Date",
        "sdo2_orientation_Last_Event_Date",
    ]
    date_keep_nat = ["sdo2_skc_SKC_DATUM"]
    date_cohort_impute = ["sdo1_previous_Final_Exam_Date"]
    cohort_col = "sdo5_degree_COLLEGEJAAR"
    
    academic_grades = [
        "sdo6_results_Average_Grade_A",
        "sdo6_results_Average_Grade_B",
    ]
    academic_other = [
        "sdo6_results_Percentage_Credits_B",
        "sdo6_results_Potential_Credits_B",
        "sdo6_results_Percentage_Credits_A",
        "sdo6_results_Potential_Credits_A",
    ]
    distance_cols = [
        "sdo1_prev_distance_distance_to_3584_CS",
        "sdo1_prev_distance_distance_to_3812_PA",
        "sdo1_postal_distance_distance_to_3584_CS",
        "sdo1_postal_distance_distance_to_3812_PA",
    ]

    # Helper functions
    def add_flag(col: str):
        """Add missing value indicator flag."""
        df[f"{col}_missing_flag"] = df[col].isna().astype("int8")

    def to_datetime(cols):
        """Convert columns to datetime."""
        for c in cols:
            if c in df.columns:
                df[c] = pd.to_datetime(df[c], errors="coerce")

    def coerce_numeric(cols):
        """Convert columns to numeric."""
        for c in cols:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

    def mode_safe(series: pd.Series):
        """Get mode or NaN if empty."""
        m = series.mode(dropna=True)
        return m.iloc[0] if len(m) > 0 else np.nan

    # Convert datatypes
    all_date_cols = orientation_date_cols + date_keep_nat + date_cohort_impute
    to_datetime(all_date_cols)
    coerce_numeric(orientation_count_cols + academic_grades + academic_other + 
                   distance_cols + [cohort_col])

    # 1) Orientation (structural missingness)
    for c in orientation_count_cols:
        if c in df.columns:
            add_flag(c)
            df[c] = df[c].fillna(0)

    for c in orientation_date_cols:
        if c in df.columns:
            add_flag(c)  # Keep NaT

    # 2) Dates
    for c in date_keep_nat:
        if c in df.columns:
            add_flag(c)  # Keep NaT

    # Cohort-wise imputation for exam dates
    for c in date_cohort_impute:
        if c in df.columns:
            add_flag(c)
            if cohort_col in df.columns:
                # Cohort-wise median
                cohort_medians = df.groupby(cohort_col)[c].transform("median")
                df[c] = df[c].fillna(cohort_medians)
            
            # Global median fallback
            if df[c].isna().any():
                global_median = df[c].median()
                if not pd.isna(global_median):
                    df[c] = df[c].fillna(global_median)

    # 3) Academic performance
    # Grades: Create Sat_Exam flags + median imputation
    for gcol in academic_grades:
        if gcol in df.columns:
            df[f"{gcol}_Sat_Exam"] = df[gcol].notna().astype("int8")
            median_grade = df[gcol].median(skipna=True)
            if not pd.isna(median_grade):
                df[gcol] = df[gcol].fillna(median_grade)

    # Other academic metrics: Fill with 0
    for c in academic_other:
        if c in df.columns:
            add_flag(c)
            df[c] = df[c].fillna(0)

    # 4) Distances: Median imputation
    for c in distance_cols:
        if c in df.columns:
            add_flag(c)
            median_dist = df[c].median(skipna=True)
            df[c] = df[c].fillna(median_dist)


    # 5) SDO78 questionnaire columns: cohort-wise median imputation
    # Treat missing response_type as students with no questionnaire response
    if "sdo78_response_type" in df.columns:
        df["sdo78_response_type"] = df["sdo78_response_type"].fillna("Non-responder")

    # sdo78_response_type is categorical and already encodes missingness — skip it
    sdo78_numeric_cols = [
        col for col in df.columns
        if col.startswith("sdo78_") and col != "sdo78_response_type" and df[col].dtype in ["float64", "int64", "object"]
        and pd.to_numeric(df[col], errors="coerce").notna().any()
    ]
    for c in sdo78_numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        if df[c].isna().any():
            if cohort_col in df.columns:
                cohort_medians = df.groupby(cohort_col)[c].transform("median")
                df[c] = df[c].fillna(cohort_medians)
            # Global median fallback for any remaining NaNs
            global_median = df[c].median(skipna=True)
            if not pd.isna(global_median):
                df[c] = df[c].fillna(global_median)

    return df


# Apply imputation
print("Applying imputation strategy...")
data = impute_all(data)
print("✅ Imputation complete")

In [ ]:
# Check remaining missing values
remaining_nan = data.isna().sum().sort_values(ascending=False)
print("Top 20 Columns with Remaining Missing Values:")
print(remaining_nan[remaining_nan > 0].head(20))

total_missing = remaining_nan.sum()
print(f"\nTotal missing values: {total_missing:,}")

## 4. Validation

Verify that imputation was successful and check remaining missing values.

In [ ]:
# Check new columns (flags and indicators)
new_cols = [col for col in data.columns if 'missing_flag' in col or 'Sat_Exam' in col]
print(f"Added {len(new_cols)} indicator columns:")
for col in sorted(new_cols):
    print(f"  - {col}")

In [ ]:
# Validate sdo78 imputation: check remaining NaN counts for sdo78 columns
sdo78_cols = [c for c in data.columns if c.startswith("sdo78_") and c != "sdo78_response_type"]
sdo78_nan = data[sdo78_cols].isna().sum()
print("SDO78 column NaN counts after imputation:")
print(sdo78_nan[sdo78_nan > 0].to_string() if sdo78_nan.any() else "  ✅ No NaN values in sdo78 numeric columns")
print(f"\nSDO78 response_type distribution:")
if "sdo78_response_type" in data.columns:
    print(data["sdo78_response_type"].value_counts(dropna=False).to_string())


In [ ]:
# Final dataset shape
print(f"Final dataset shape: {data.shape}")
print(f"Rows: {data.shape[0]:,}, Columns: {data.shape[1]:,}")

## 5. Export

Save the processed dataset for the next pipeline stage.

In [ ]:
# Save processed dataset
output_path = OUT_DIR / "missing_values_imputed.csv"
data.to_csv(output_path, index=False)
print(f"✅ Dataset saved to: {output_path}")
print(f"   Shape: {data.shape}")